# FLUKE Sentiment Analysis with DeepSeek R1

This notebook evaluates sentiment analysis robustness using DeepSeek R1 open-source reasoning model via OpenRouter API with FLUKE linguistic modifications.

In [53]:
# Standard imports
from datasets import load_dataset
import dspy
import os
import pandas as pd
import json
import glob
import time
from dotenv import load_dotenv
from dspy.evaluate import Evaluate

# Import unified FLUKE utilities
from fluke_reasoning_utils import (
    REASONING_MODELS, REASONING_CONFIGS,
    remove_space, extract_classification_prediction,
    aggregate_results, highlight_drops_and_significance,
    compare_models
)

In [54]:
# Load environment variables
load_dotenv()

# For OpenRouter, we need the OpenRouter API key
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
if not openrouter_api_key:
    print("Warning: OPENROUTER_API_KEY not found in environment variables")
    print("Please set your OpenRouter API key in the .env file")

## DeepSeek R1 Configuration

In [55]:
# Select DeepSeek configuration
CONFIG_NAME = 'deepseek'  # Options: 'deepseek', 'deepseek-lite'
config = REASONING_CONFIGS[CONFIG_NAME]

MODEL_NAME = config['model']
MODEL_ID = REASONING_MODELS[MODEL_NAME]

print(f"Configuration: {CONFIG_NAME}")
print(f"Model: {MODEL_NAME} ({MODEL_ID})")
print(f"Description: {config['description']}")

# Configure DSPy with DeepSeek R1 via OpenRouter
# DeepSeek R1 supports various parameters
lm = dspy.LM(
    model=MODEL_ID,
    api_key=openrouter_api_key,
    api_base="https://openrouter.ai/api/v1",
    max_tokens=20_001,
    temperature=1  # DeepSeek R1 supports temperature control
)
dspy.configure(lm=lm)

Configuration: deepseek
Model: deepseek-r1 (openrouter/deepseek/deepseek-r1)
Description: Open-source reasoning with DeepSeek R1


## Load Data

In [56]:
# Load sentiment dataset from JSON file
with open('../../../data/train_dev_test_data/sent/test.json', 'r') as f:
    ds = json.load(f)
print(f"Dataset size: {len(ds)}")

# Create examples
examples = [
    dspy.Example({
        "text": remove_space(item["sentence"]),
        "label": item["label"]
    }).with_inputs("text")
    for item in ds
]

# Test example
example = examples[0]
print(f"\nExample text: {example.text}")
print(f"Label: {example.label}")

Dataset size: 872

Example text: it's a charming and often affecting journey.
Label: 1


## Define Task with DeepSeek R1

In [57]:
class DeepSeekSentiment(dspy.Signature):
    """Classify sentiment of the given text. Analyze the emotional tone, word choice, and overall sentiment. Make sure to answer with 1 for positive sentiment, 0 for negative sentiment. Only answer with 1 or 0."""
    text = dspy.InputField()
    label = dspy.OutputField(prefix='Answer:')

class DeepSeekSentimentModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.Predict(DeepSeekSentiment)

    def forward(self, text):
        # Try up to 3 times to get a valid prediction
        for _ in range(3):
            pred = self.prog(text=text)
            parsed = extract_classification_prediction(pred.label)
            if parsed in ['0', '1']:
                return pred
        # If still no valid prediction after retries, return last attempt
        return pred

# Initialize module
deepseek_sentiment = DeepSeekSentimentModule()

# Evaluation metric
def eval_metric(true, prediction, trace=None):
    pred = prediction.label
    # print(pred)
    parsed_answer = extract_classification_prediction(pred)
    return parsed_answer == str(true.label)

In [58]:
# Test single example
pred = deepseek_sentiment(text=example.text)
print(f"Text: {example.text}")
print(f"True Label: {example.label}")
print(f"Prediction: {pred.label}")
print(f"Correct: {eval_metric(example, pred)}")

Text: it's a charming and often affecting journey.
True Label: 1
Prediction: 1
Correct: True


## Evaluate Original Dataset

In [59]:
# Test size for DeepSeek R1
TEST_SIZE = 200  # Adjust based on API limits and budget
test_examples = examples

print(f"Evaluating {len(test_examples)} examples with DeepSeek R1...")

evaluate = Evaluate(
    devset=test_examples,
    metric=eval_metric,
    num_threads=2,  # Moderate threading for OpenRouter
    display_progress=True,
    display_table=10,
    return_all_scores=True,
)

results = evaluate(deepseek_sentiment)

# Save results
items = []
for sample in results['results']:
    items.append({
        'text': sample[0]['text'],
        'label': sample[0]['label'],
        'pred': extract_classification_prediction(sample[1]['label']),
        'raw_output': sample[1]['label']
    })

df_result = pd.DataFrame(items)
output_file = f'../results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
df_result.to_csv(output_file, index=False)

print(f"\nDeepSeek R1 Accuracy: {results['score']:.3f}")
print(f"Results saved to: {output_file}")

Evaluating 872 examples with DeepSeek R1...
Average Metric: 6.00 / 6 (100.0%):   1%|          | 5/872 [00:00<00:48, 17.72it/s]

Average Metric: 11.00 / 11 (100.0%):   1%|          | 10/872 [00:00<00:57, 15.05it/s]

2025/08/19 14:54:48 ERROR dspy.utils.parallelizer: Error for Example({'text': "if the movie succeeds in instilling a wary sense of ` there but for the grace of god, ' it is far too self-conscious to draw you deeply into its world.", 'label': 0}) (input_keys={'text'}): expected string or bytes-like object, got 'NoneType'. Set `provide_traceback=True` for traceback.


Average Metric: 110.00 / 117 (94.0%):  13%|█▎        | 117/872 [00:02<00:12, 62.04it/s]

2025/08/19 14:54:49 ERROR dspy.utils.parallelizer: Error for Example({'text': "it all drags on so interminably it's like watching a miserable relationship unfold in real time.", 'label': 0}) (input_keys={'text'}): expected string or bytes-like object, got 'NoneType'. Set `provide_traceback=True` for traceback.


...

Average Metric: 462.00 / 497 (93.0%):  57%|█████▋    | 500/872 [00:12<00:13, 28.00it/s]

2025/08/19 14:55:00 ERROR dspy.utils.parallelizer: Error for Example({'text': "a working class `` us vs. them '' opera that leaves no heartstring untugged and no liberal cause unplundered.", 'label': 1}) (input_keys={'text'}): expected string or bytes-like object, got 'NoneType'. Set `provide_traceback=True` for traceback.


Average Metric: 496.00 / 533 (93.1%):  62%|██████▏   | 537/872 [00:12<00:07, 42.00it/s]

2025/08/19 14:55:00 WARNING dspy.utils.parallelizer: Execution cancelled due to errors or interruption.


Exception: Execution cancelled due to errors or interruption.

## Evaluate Modifications

In [ ]:
def evaluate_modified_set(data, program, max_samples=50):
    """Evaluate on modified dataset with DeepSeek R1."""
    limited_data = data[:max_samples] if len(data) > max_samples else data
    
    mod_examples = [
        dspy.Example({
            "text": remove_space(r['modified_text']),
            "original_text": remove_space(r['original_text']),
            "label": int(r.get('modified_label', r['label'])),
            "original_label": int(r['label'])
        }).with_inputs("text")
        for r in limited_data
    ]
    
    evaluate = Evaluate(
        devset=mod_examples,
        metric=eval_metric,
        num_threads=2,  # Moderate threading for OpenRouter
        display_progress=True,
        display_table=1,
        return_outputs=True,
        return_all_scores=True
    )
    
    return evaluate(program)

In [ ]:
# Load original predictions
original_pred_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv'
if os.path.exists(original_pred_file):
    original_pred_ds = pd.read_csv(original_pred_file)
    original_pred_ds['text'] = original_pred_ds['text'].apply(remove_space)
    print(f"Loaded original DeepSeek R1 predictions from {original_pred_file}")
else:
    print("Please run original evaluation first")
    original_pred_ds = None

# Test modifications with DeepSeek R1
json_files = glob.glob('../data/modified_data/sa/*_100.json')
# Test a subset of modifications
test_modifications = [
    'typo_bias_100.json', 'capitalization_100.json', 'punctuation_100.json',
    'negation_100.json', 'sentiment_100.json', 'active_to_passive_100.json'
]
json_files = [f for f in json_files if any(mod in f for mod in test_modifications)]

print(f"\nTesting {len(json_files)} modifications with DeepSeek R1...")

for json_file in json_files:
    print(f"\nProcessing: {json_file.split('/')[-1]}")
    
    with open(json_file, 'r') as f:
        data = json.load(f)
    
    # Evaluate with sample limit
    results_mod = evaluate_modified_set(data, deepseek_sentiment, max_samples=50)
    
    # Process results
    items = []
    for sample in results_mod[1]:
        item = {
            'text': sample[0]['text'],
            'original_text': sample[0]['original_text'],
            'modified_label': sample[0]['label'],
            'original_label': sample[0]['original_label'],
            'modified_pred': extract_classification_prediction(sample[1]['label']),
            'raw_output': sample[1]['label']
        }
        
        # Find original prediction
        if original_pred_ds is not None:
            matches = original_pred_ds[original_pred_ds['text'] == item['original_text']]
            item['original_pred'] = matches.iloc[0]['pred'] if not matches.empty else None
        else:
            item['original_pred'] = None
        
        items.append(item)
    
    df_mod = pd.DataFrame(items)
    mod_name = json_file.split('/')[-1].replace('.json', '')
    output_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-{mod_name}.csv'
    df_mod.to_csv(output_file, index=False)
    
    print(f"Accuracy: {results_mod[0]:.3f}")
    print(f"Saved to: {output_file}")
    
    time.sleep(3)  # Rate limiting for OpenRouter

## Chain-of-Thought with DeepSeek R1

In [ ]:
class CoTDeepSeekSentiment(dspy.Module):
    def __init__(self):
        super().__init__()
        self.prog = dspy.ChainOfThought(DeepSeekSentiment)

    def forward(self, text):
        return self.prog(text=text)

# Test CoT
cot_deepseek_sentiment = CoTDeepSeekSentiment()
pred_cot = cot_deepseek_sentiment(text=example.text)
print("Chain-of-Thought with DeepSeek R1:")
print(f"Text: {example.text}")
print(f"\nReasoning: {pred_cot.reasoning if hasattr(pred_cot, 'reasoning') else 'N/A'}")
print(f"\nPrediction: {pred_cot.label}")

## Aggregate Results

In [ ]:
# Aggregate all modification results
result_files = glob.glob(f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-*_100.csv')

if result_files:
    results_df = aggregate_results(
        result_files,
        task_name='sentiment_analysis',
        model_name=f'{MODEL_NAME}-{CONFIG_NAME}'
    )
    
    if not results_df.empty:
        # Display summary
        print(f"\n{MODEL_NAME}-{CONFIG_NAME} Results Summary:")
        print(results_df[['modification', 'original_res', 'modified_res', 'difference', 'samples']])
        
        # Save aggregated results
        output_file = f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-DP.csv'
        results_df.to_csv(output_file, index=False)
        print(f"\nAggregated results saved to: {output_file}")
        
        # Display styled results
        styled_df = results_df.round(3).style.apply(highlight_drops_and_significance, axis=1)
        display(styled_df)
else:
    print("No result files found to aggregate")

## Model Comparison

In [ ]:
# Compare DeepSeek R1 with other models
comparison_files = {
    'DeepSeek-R1': f'results/sa/{MODEL_NAME}-{CONFIG_NAME}-0shot-sst2.csv',
    'GPT-5': 'results/sa/gpt-5-standard-0shot-sst2.csv',
    'GPT-4o': 'results/sa/gpt4o-0shot-sst2.csv',
    'Claude-3.5': 'results/sa/claude-3-5-sonnet-0shot-sst2.csv',
    'o3-2025-04-16': 'results/sa/o3-2025-04-16-standard-0shot-sst2.csv',
    'Mixtral-8x22B': 'results/sa/mixtral-8x22b-sst2.csv'
}

comparison_df = compare_models(comparison_files, task_name='sentiment_analysis')

if not comparison_df.empty:
    print("\nModel Comparison (including DeepSeek R1):")
    print(comparison_df)
    
    # Calculate DeepSeek R1 performance relative to others
    if 'DeepSeek-R1' in comparison_df['Model'].values:
        deepseek_acc = comparison_df[comparison_df['Model'] == 'DeepSeek-R1']['Accuracy'].values[0]
        
        # Compare with closed-source models
        closed_models = ['GPT-5', 'GPT-4o', 'Claude-3.5', 'o3-2025-04-16']
        closed_accs = comparison_df[comparison_df['Model'].isin(closed_models)]['Accuracy'].values
        
        if len(closed_accs) > 0:
            avg_closed = closed_accs.mean()
            gap = deepseek_acc - avg_closed
            print(f"\nDeepSeek R1 Performance: {deepseek_acc:.3f}")
            print(f"Average of closed-source models: {avg_closed:.3f}")
            print(f"Performance gap: {gap:+.3f} ({gap*100:+.1f}%)")
            print(f"\nNote: DeepSeek R1 is an open-source model competing with proprietary systems")
    
    # Highlight best performer
    def highlight_max(s):
        is_max = s == s.max()
        return ['background-color: green; color: white' if v else '' for v in is_max]
    
    styled_comparison = comparison_df.style.apply(highlight_max, subset=['Accuracy'])
    display(styled_comparison)
else:
    print("No comparison data available")

## DeepSeek R1 Performance Analysis

In [ ]:
print(f"\n{'='*60}")
print(f"FLUKE Sentiment Analysis with DeepSeek R1 Complete!")
print(f"{'='*60}")

if 'results' in locals():
    print(f"\nBase accuracy: {results[0]:.3f}")

if 'results_df' in locals() and not results_df.empty:
    avg_row = results_df[results_df['modification'] == 'average'].iloc[0]
    print(f"Average robustness drop: {avg_row['difference']:.3f}")
    print(f"Modifications tested: {len(results_df) - 1}")

print(f"\nDeepSeek R1 Configuration: {config['description']}")
print(f"\nKey features of DeepSeek R1:")
print("• Open-source reasoning model")
print("• Competitive performance with proprietary models")
print("• Cost-effective via OpenRouter API")
print("• Supports temperature and system messages")
print("• Strong chain-of-thought capabilities")

print(f"\nFiles saved in: results/sa/")
print(f"\nOpenRouter API endpoint: https://openrouter.ai/api/v1")
print(f"Model ID: {MODEL_ID}")